In [30]:
print("Spark session is working")

Spark session is working


In [31]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [1]:
BUCKET = "mahmoud-sic-ecommerce-2026"

RAW_PATH = f"s3://{BUCKET}/raw/ecommerce"
PROCESSED_PATH = f"s3://{BUCKET}/processed/ecommerce"


Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Trying to create a Glue session for the kernel.
Session Type: glueetl
Session ID: 6c168906-d72f-4f8f-8061-a3b275ceacb4
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session 6c168906-d72f-4f8f-8061-a3b275ceacb4 to get into ready status...
Session 6c168906-d72f-4f8f-8061-a3b275ceacb4 has been created.



In [2]:
# 2. Read all raw CSV files
customers = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/customers.csv")
)

categories = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/categories.csv")
)

products = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/products.csv")
)

departments = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/departments.csv")
)

employees = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/employees.csv")
)

suppliers = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/suppliers.csv")
)

orders = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/orders.csv")
)

order_details = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/order_details.csv")
)

payments = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/payments.csv")
)

product_suppliers = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/product_suppliers.csv")
)

shippers = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/shippers.csv")
)

shipments = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/shipments.csv")
)

In [3]:
print("Customers:", customers.count())
print("Categories:", categories.count())
print("Products:", products.count())
print("Departments:", departments.count())
print("Employees:", employees.count())
print("Suppliers:", suppliers.count())
print("Orders:", orders.count())
print("Order Details:", order_details.count())
print("Payments:", payments.count())
print("Product Suppliers:", product_suppliers.count())
print("Shippers:", shippers.count())
print("Shipments:", shipments.count())

Customers: 10000
Categories: 20
Products: 1000
Departments: 10
Employees: 200
Suppliers: 100
Orders: 50000
Order Details: 100000
Payments: 45000
Product Suppliers: 2027
Shippers: 10
Shipments: 40000


In [4]:
# Part 1 - Profiling all tables

tables = {
    "customers": customers,
    "categories": categories,
    "products": products,
    "departments": departments,
    "employees": employees,
    "suppliers": suppliers,
    "orders": orders,
    "order_details": order_details,
    "payments": payments,
    "product_suppliers": product_suppliers,
    "shippers": shippers,
    "shipments": shipments
}

for table_name, df in tables.items():
    print("=" * 80)
    print(f"TABLE: {table_name}")
    print("- Schema:")
    df.printSchema()
    print("- First 10 records:")
    df.show(10, truncate=False)


TABLE: customers
- Schema:
root
 |-- CustomerID: integer (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- RegistrationDate: timestamp (nullable = true)

- First 10 records:
+----------+---------+--------+-----------------------------+----------+--------------+-------------------+
|CustomerID|FirstName|LastName|Email                        |City      |Country       |RegistrationDate   |
+----------+---------+--------+-----------------------------+----------+--------------+-------------------+
|1         |Danielle |Johnson |danielle.johnson1@example.com|Giza      |Egypt         |2023-01-31 00:00:00|
|2         |Joshua   |Walker  |joshua.walker2@example.com   |Dubai     |Jordan        |2021-07-25 00:00:00|
|3         |Jill     |Rhodes  |jill.rhodes3@example.com     |Giza      |United Kingdom|2020-12-22 00:00:00|
|4         |

In [12]:
from pyspark.sql import functions as F

# NULL values in every column of customers
customer_nulls = customers.select([
    F.sum(F.col(column_name).isNull().cast("int")).alias(column_name)
    for column_name in customers.columns
])

print("NULL values in customers:")
customer_nulls.show(truncate=False)

# Duplicate customer IDs
duplicate_customers = (
    customers
    .groupBy("CustomerID")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate customerid values:")
duplicate_customers.show(truncate=False)

# Duplicate order IDs
duplicate_orders = (
    orders
    .groupBy("OrderID")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate orderid values:")
duplicate_orders.show(truncate=False)


NULL values in customers:
+----------+---------+--------+-----+----+-------+----------------+
|CustomerID|FirstName|LastName|Email|City|Country|RegistrationDate|
+----------+---------+--------+-----+----+-------+----------------+
|0         |0        |0       |0    |0   |0      |0               |
+----------+---------+--------+-----+----+-------+----------------+

Duplicate customerid values:
+----------+-----+
|CustomerID|count|
+----------+-----+
+----------+-----+

Duplicate orderid values:
+-------+-----+
|OrderID|count|
+-------+-----+
+-------+-----+


In [13]:
# Part 2 - Data Cleaning

from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# Put all tables in a dictionary
tables = {
    "customers": customers,
    "categories": categories,
    "products": products,
    "departments": departments,
    "employees": employees,
    "suppliers": suppliers,
    "orders": orders,
    "order_details": order_details,
    "payments": payments,
    "product_suppliers": product_suppliers,
    "shippers": shippers,
    "shipments": shipments
}

# 1. Standardize all column names to lowercase
def standardize_column_names(df):
    for old_name in df.columns:
        new_name = old_name.strip().lower().replace(" ", "_").replace("-", "_")
        df = df.withColumnRenamed(old_name, new_name)
    return df

for table_name in tables:
    tables[table_name] = standardize_column_names(tables[table_name])

# Update the individual DataFrame variables
customers = tables["customers"]
categories = tables["categories"]
products = tables["products"]
departments = tables["departments"]
employees = tables["employees"]
suppliers = tables["suppliers"]
orders = tables["orders"]
order_details = tables["order_details"]
payments = tables["payments"]
product_suppliers = tables["product_suppliers"]
shippers = tables["shippers"]
shipments = tables["shipments"]

# 2. Trim leading/trailing spaces from all string columns
for table_name in tables:
    df = tables[table_name]

    for field in df.schema.fields:
        if isinstance(field.dataType, StringType):
            df = df.withColumn(field.name, F.trim(F.col(field.name)))

    tables[table_name] = df

# Update DataFrame variables again
customers = tables["customers"]
categories = tables["categories"]
products = tables["products"]
departments = tables["departments"]
employees = tables["employees"]
suppliers = tables["suppliers"]
orders = tables["orders"]
order_details = tables["order_details"]
payments = tables["payments"]
product_suppliers = tables["product_suppliers"]
shippers = tables["shippers"]
shipments = tables["shipments"]

# 3. Convert customer emails to lowercase
customers = customers.withColumn(
    "email",
    F.lower(F.col("email"))
)

# 4. Convert order statuses to uppercase
orders = orders.withColumn(
    "status",
    F.upper(F.col("status"))
)

# 5. Find invalid customer emails
invalid_emails = customers.filter(
    ~F.col("email").rlike(
        r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
    )
)

print("Invalid customer emails:")
invalid_emails.show(10, truncate=False)
print("Invalid email count:", invalid_emails.count())

# 6. Find invalid order statuses
valid_statuses = [
    "PENDING",
    "SHIPPED",
    "DELIVERED",
    "CANCELLED"
]

invalid_statuses = orders.filter(
    ~F.col("status").isin(valid_statuses)
)

print("Invalid order statuses:")
invalid_statuses.groupBy("status").count().show(truncate=False)

# 7. Find products with NULL or non-positive prices
invalid_products = products.filter(
    F.col("price").isNull() | (F.col("price") <= 0)
)

print("Products with NULL or non-positive prices:")
invalid_products.show(10, truncate=False)
print("Invalid product count:", invalid_products.count())

# 8. Remove duplicate records based on primary keys
customers = customers.dropDuplicates(["customerid"])
orders = orders.dropDuplicates(["orderid"])

# 9. Remove records with NULL primary keys
primary_keys = {
    "customers": "customerid",
    "categories": "categoryid",
    "products": "productid",
    "departments": "departmentid",
    "employees": "employeeid",
    "suppliers": "supplierid",
    "orders": "orderid",
    "order_details": "orderdetailid",
    "payments": "paymentid",
    "product_suppliers": "productsupplierid",
    "shippers": "shipperid",
    "shipments": "shipmentid"
}

for table_name, primary_key in primary_keys.items():
    if primary_key in tables[table_name].columns:
        tables[table_name] = tables[table_name].filter(
            F.col(primary_key).isNotNull()
        )

# Update cleaned tables
customers = tables["customers"]
categories = tables["categories"]
products = tables["products"]
departments = tables["departments"]
employees = tables["employees"]
suppliers = tables["suppliers"]
orders = tables["orders"]
order_details = tables["order_details"]
payments = tables["payments"]
product_suppliers = tables["product_suppliers"]
shippers = tables["shippers"]
shipments = tables["shipments"]

print("Data cleaning completed successfully.")


Invalid customer emails:
+----------+---------+--------+-----+----+-------+----------------+
|customerid|firstname|lastname|email|city|country|registrationdate|
+----------+---------+--------+-----+----+-------+----------------+
+----------+---------+--------+-----+----+-------+----------------+

Invalid email count: 0
Invalid order statuses:
+----------+-----+
|status    |count|
+----------+-----+
|PROCESSING|9968 |
+----------+-----+

Products with NULL or non-positive prices:
+---------+----------+-----------+-----+-----+----+-----+
|productid|categoryid|productname|brand|price|cost|stock|
+---------+----------+-----------+-----+-----+----+-----+
+---------+----------+-----------+-----+-----+----+-----+

Invalid product count: 0
Data cleaning completed successfully.


In [14]:
# Part 3 - Data Type Transformation

from pyspark.sql import functions as F

# 1. Convert all ID columns to long
for table_name, df in tables.items():
    for column_name in df.columns:
        if column_name.endswith("id"):
            df = df.withColumn(
                column_name,
                F.col(column_name).cast("long")
            )
    tables[table_name] = df

# 2. Convert price and money columns to decimal(12,2)

products = products \
    .withColumn("price", F.col("price").cast("decimal(12,2)")) \
    .withColumn("cost", F.col("cost").cast("decimal(12,2)"))

order_details = order_details \
    .withColumn("unitprice", F.col("unitprice").cast("decimal(12,2)")) \
    .withColumn("discount", F.col("discount").cast("decimal(12,2)"))

payments = payments.withColumn(
    "amount",
    F.col("amount").cast("decimal(12,2)")
)

# 3. Convert date columns to Spark date
customers = customers.withColumn(
    "registrationdate",
    F.to_date("registrationdate")
)

employees = employees.withColumn(
    "hiredate",
    F.to_date("hiredate")
)

orders = orders.withColumn(
    "orderdate",
    F.to_date("orderdate")
)

payments = payments.withColumn(
    "paymentdate",
    F.to_date("paymentdate")
)

shipments = shipments \
    .withColumn("shipdate", F.to_date("shipdate")) \
    .withColumn("deliverydate", F.to_date("deliverydate"))

# 4. Convert quantity to integer
order_details = order_details.withColumn(
    "quantity",
    F.col("quantity").cast("int")
)

# 5. Extract date parts from orderdate
orders = orders \
    .withColumn("year", F.year("orderdate")) \
    .withColumn("month", F.month("orderdate")) \
    .withColumn("quarter", F.quarter("orderdate")) \
    .withColumn("day", F.dayofmonth("orderdate")) \
    .withColumn("day_of_week", F.dayofweek("orderdate"))

# 6. Recalculate total_amount after converting quantity and unitprice
order_details = order_details.withColumn(
    "total_amount",
    (
        F.col("quantity").cast("decimal(12,2)") *
        F.col("unitprice")
    ).cast("decimal(12,2)")
)

# 7. Update the tables dictionary
tables["customers"] = customers
tables["products"] = products
tables["orders"] = orders
tables["order_details"] = order_details
tables["payments"] = payments
tables["employees"] = employees
tables["shipments"] = shipments

print("Data type transformation completed successfully.")


Data type transformation completed successfully.


In [15]:
print("CUSTOMERS SCHEMA")
customers.printSchema()

print("ORDERS SCHEMA")
orders.printSchema()

print("ORDER_DETAILS SCHEMA")
order_details.printSchema()

print("PAYMENTS SCHEMA")
payments.printSchema()


CUSTOMERS SCHEMA
root
 |-- customerid: integer (nullable = true)
 |-- firstname: string (nullable = true)
 |-- lastname: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- registrationdate: date (nullable = true)

ORDERS SCHEMA
root
 |-- orderid: integer (nullable = true)
 |-- customerid: integer (nullable = true)
 |-- orderdate: date (nullable = true)
 |-- status: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)

ORDER_DETAILS SCHEMA
root
 |-- orderdetailid: integer (nullable = true)
 |-- orderid: integer (nullable = true)
 |-- productid: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unitprice: decimal(12,2) (nullable = true)
 |-- discount: decimal(12,2) (nullable = true)
 |-- total_amount: decimal(12,

In [17]:
# Part 4 - Business Transformations

from pyspark.sql import functions as F

# 1. Create full_name in customers
customers = customers.withColumn(
    "full_name",
    F.concat_ws(" ", F.col("firstname"), F.col("lastname"))
)

# 2. Calculate total amount for every order detail
order_details = order_details.withColumn(
    "total_amount",
    (
        F.col("quantity").cast("decimal(12,2)") *
        F.col("unitprice")
    ).cast("decimal(12,2)")
)

# 3. Create the sales fact dataset
fact_sales = (
    orders.alias("o")
    .join(
        order_details.alias("od"),
        F.col("o.orderid") == F.col("od.orderid"),
        "inner"
    )
    .join(
        products.alias("p"),
        F.col("od.productid") == F.col("p.productid"),
        "left"
    )
    .join(
        categories.alias("c"),
        F.col("p.categoryid") == F.col("c.categoryid"),
        "left"
    )
    .join(
        customers.alias("cu"),
        F.col("o.customerid") == F.col("cu.customerid"),
        "left"
    )
    .select(
        F.col("o.orderid").alias("orderid"),
        F.col("o.customerid").alias("customerid"),
        F.col("od.productid").alias("productid"),
        F.col("p.categoryid").alias("categoryid"),
        F.col("o.orderdate").alias("orderdate"),
        F.col("o.year").alias("year"),
        F.col("o.month").alias("month"),
        F.col("o.quarter").alias("quarter"),
        F.col("o.day").alias("day"),
        F.col("o.day_of_week").alias("day_of_week"),
        F.col("od.quantity").alias("quantity"),
        F.col("od.unitprice").alias("unitprice"),
        F.col("od.total_amount").alias("total_amount"),
        F.col("o.status").alias("status"),
        F.col("p.productname").alias("productname"),
        F.col("c.categoryname").alias("categoryname"),
        F.col("cu.full_name").alias("customer_name"),
        F.col("cu.city").alias("city"),
        F.col("cu.country").alias("country")
    )
)

# 4. Calculate total amount and total quantity for every order
order_totals = (
    fact_sales
    .groupBy("orderid")
    .agg(
        F.sum("total_amount")
         .cast("decimal(12,2)")
         .alias("order_total"),
        F.sum("quantity")
         .cast("long")
         .alias("order_quantity")
    )
)

# 5. Add order totals and order value category to orders
orders = (
    orders
    .join(order_totals, on="orderid", how="left")
    .withColumn(
        "order_value_category",
        F.when(F.col("order_total") >= 1000, "HIGH")
         .when(F.col("order_total") >= 500, "MEDIUM")
         .otherwise("LOW")
    )
)

# 6. Calculate total sales for every customer
customer_sales_base = (
    fact_sales
    .groupBy("customerid")
    .agg(
        F.sum("total_amount")
         .cast("decimal(14,2)")
         .alias("total_sales"),
        F.countDistinct("orderid").alias("order_count"),
        F.sum("quantity").cast("long").alias("total_quantity")
    )
)

# 7. Create customer segments
# Assumption documented because the assignment did not define numeric thresholds.
customer_sales = (
    customers
    .select("customerid", "full_name", "city", "country")
    .join(customer_sales_base, on="customerid", how="left")
    .fillna({
        "total_sales": 0,
        "order_count": 0,
        "total_quantity": 0
    })
    .withColumn(
        "customer_segment",
        F.when(F.col("total_sales") >= 10000, "VIP")
         .when(F.col("total_sales") >= 5000, "PREMIUM")
         .when(F.col("total_sales") >= 1000, "REGULAR")
         .otherwise("LOW_VALUE")
    )
)

# 8. Average product price
average_product_price = products.select(
    F.avg("price").cast("decimal(12,2)").alias("average_product_price")
)

print("Business transformations completed successfully.")
print("fact_sales rows:", fact_sales.count())
print("customer_sales rows:", customer_sales.count())

print("Average product price:")
average_product_price.show()


Business transformations completed successfully.
fact_sales rows: 100000
customer_sales rows: 10000
Average product price:
+---------------------+
|average_product_price|
+---------------------+
|              1530.10|
+---------------------+


In [35]:
customers.printSchema()
orders.printSchema()
order_details.printSchema()
products.printSchema()

root
 |-- CustomerID: integer (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- RegistrationDate: timestamp (nullable = true)

root
 |-- OrderID: integer (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- OrderDate: timestamp (nullable = true)
 |-- Status: string (nullable = true)

root
 |-- OrderDetailID: integer (nullable = true)
 |-- OrderID: integer (nullable = true)
 |-- ProductID: integer (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- Discount: integer (nullable = true)

root
 |-- ProductID: integer (nullable = true)
 |-- CategoryID: integer (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Brand: string (nullable = true)
 |-- Price: double (nullable = true)
 |-- Cost: double (nullable = true)
 |-- Stock: integer (nullable = true

In [18]:
# Part 5 - Window Functions

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Latest order for every customer
latest_order_window = (
    Window
    .partitionBy("customerid")
    .orderBy(
        F.col("orderdate").desc(),
        F.col("orderid").desc()
    )
)

latest_orders = (
    orders
    .withColumn(
        "row_number",
        F.row_number().over(latest_order_window)
    )
    .filter(F.col("row_number") == 1)
    .select(
        "customerid",
        "orderid",
        "status",
        "orderdate"
    )
)

# 2. First order for every customer
first_order_window = (
    Window
    .partitionBy("customerid")
    .orderBy(
        F.col("orderdate").asc(),
        F.col("orderid").asc()
    )
)

first_orders = (
    orders
    .withColumn(
        "row_number",
        F.row_number().over(first_order_window)
    )
    .filter(F.col("row_number") == 1)
    .select(
        "customerid",
        "orderid",
        "status",
        "orderdate"
    )
)

# 3. Rank customers by total sales
customer_rank_window = Window.orderBy(
    F.col("total_sales").desc()
)

ranked_customers = (
    customer_sales
    .withColumn(
        "sales_rank",
        F.rank().over(customer_rank_window)
    )
)

# 4. Top 3 products in every category by sales
product_sales_by_category = (
    fact_sales
    .groupBy(
        "categoryid",
        "categoryname",
        "productid",
        "productname"
    )
    .agg(
        F.sum("total_amount")
         .cast("decimal(14,2)")
         .alias("product_sales")
    )
)

top_products_window = (
    Window
    .partitionBy("categoryid")
    .orderBy(F.col("product_sales").desc())
)

top_3_products_per_category = (
    product_sales_by_category
    .withColumn(
        "category_rank",
        F.row_number().over(top_products_window)
    )
    .filter(F.col("category_rank") <= 3)
)

# 5. Most expensive product in every category
most_expensive_window = (
    Window
    .partitionBy("categoryid")
    .orderBy(
        F.col("price").desc(),
        F.col("productid").asc()
    )
)

most_expensive_products = (
    products
    .join(
        categories,
        on="categoryid",
        how="left"
    )
    .withColumn(
        "price_rank",
        F.row_number().over(most_expensive_window)
    )
    .filter(F.col("price_rank") == 1)
    .select(
        "categoryid",
        "categoryname",
        "productid",
        "productname",
        "price"
    )
)

# 6. Latest shipment for every order
latest_shipment_window = (
    Window
    .partitionBy("orderid")
    .orderBy(
        F.col("shipdate").desc(),
        F.col("shipmentid").desc()
    )
)

latest_shipments = (
    shipments
    .withColumn(
        "row_number",
        F.row_number().over(latest_shipment_window)
    )
    .filter(F.col("row_number") == 1)
    .drop("row_number")
)

print("Window functions completed successfully.")

print("Latest orders per customer:", latest_orders.count())
print("First orders per customer:", first_orders.count())
print("Top 3 products per category:", top_3_products_per_category.count())
print("Most expensive products per category:", most_expensive_products.count())
print("Latest shipments per order:", latest_shipments.count())

print("Top 10 customers by sales:")
ranked_customers.orderBy("sales_rank").show(10, truncate=False)


Window functions completed successfully.
Latest orders per customer: 9931
First orders per customer: 9931
Top 3 products per category: 60
Most expensive products per category: 20
Latest shipments per order: 27459
Top 10 customers by sales:
+----------+----------------+---------+--------------+-----------+-----------+--------------+----------------+----------+
|customerid|full_name       |city     |country       |total_sales|order_count|total_quantity|customer_segment|sales_rank|
+----------+----------------+---------+--------------+-----------+-----------+--------------+----------------+----------+
|6772      |Angela Stewart  |Abu Dhabi|United Kingdom|368984.90  |12         |211           |VIP             |1         |
|245       |Amy Morrison    |Berlin   |Jordan        |338329.93  |9          |192           |VIP             |2         |
|9895      |Benjamin Holder |Jeddah   |France        |330803.47  |7          |198           |VIP             |3         |
|5446      |Lisa Hogan      

In [19]:
# Part 6 - Joins

from pyspark.sql import functions as F

# 1. Orders + Customers
orders_customers = (
    orders.alias("o")
    .join(
        customers.alias("c"),
        F.col("o.customerid") == F.col("c.customerid"),
        "left"
    )
    .select(
        F.col("o.orderid").alias("orderid"),
        F.col("o.customerid").alias("customerid"),
        F.col("c.full_name").alias("customer_name"),
        F.col("c.city").alias("city"),
        F.col("c.country").alias("country"),
        F.col("o.orderdate").alias("orderdate"),
        F.col("o.status").alias("status")
    )
)

# 2. Orders + Order Details + Products
sales_dataset = (
    orders.alias("o")
    .join(
        order_details.alias("od"),
        F.col("o.orderid") == F.col("od.orderid"),
        "inner"
    )
    .join(
        products.alias("p"),
        F.col("od.productid") == F.col("p.productid"),
        "left"
    )
    .select(
        F.col("o.orderid").alias("orderid"),
        F.col("o.customerid").alias("customerid"),
        F.col("o.orderdate").alias("orderdate"),
        F.col("o.status").alias("status"),
        F.col("od.orderdetailid").alias("orderdetailid"),
        F.col("od.productid").alias("productid"),
        F.col("od.quantity").alias("quantity"),
        F.col("od.unitprice").alias("unitprice"),
        F.col("od.total_amount").alias("total_amount"),
        F.col("p.categoryid").alias("categoryid"),
        F.col("p.productname").alias("productname")
    )
)

# 3. Products + Categories
products_categories = (
    products.alias("p")
    .join(
        categories.alias("c"),
        F.col("p.categoryid") == F.col("c.categoryid"),
        "left"
    )
    .select(
        F.col("p.productid").alias("productid"),
        F.col("p.productname").alias("productname"),
        F.col("p.categoryid").alias("categoryid"),
        F.col("c.categoryname").alias("categoryname"),
        F.col("p.brand").alias("brand"),
        F.col("p.price").alias("price"),
        F.col("p.cost").alias("cost"),
        F.col("p.stock").alias("stock")
    )
)

# 4. Products + Product Suppliers + Suppliers
products_suppliers = (
    products.alias("p")
    .join(
        product_suppliers.alias("ps"),
        F.col("p.productid") == F.col("ps.productid"),
        "left"
    )
    .join(
        suppliers.alias("s"),
        F.col("ps.supplierid") == F.col("s.supplierid"),
        "left"
    )
    .select(
        F.col("p.productid").alias("productid"),
        F.col("p.productname").alias("productname"),
        F.col("ps.supplierid").alias("supplierid"),
        F.col("s.suppliername").alias("suppliername"),
        F.col("s.country").alias("supplier_country")
    )
)

# 5. Orders + Payments
orders_payments = (
    orders.alias("o")
    .join(
        payments.alias("pay"),
        F.col("o.orderid") == F.col("pay.orderid"),
        "left"
    )
    .select(
        F.col("o.orderid").alias("orderid"),
        F.col("o.customerid").alias("customerid"),
        F.col("o.orderdate").alias("orderdate"),
        F.col("o.status").alias("status"),
        F.col("pay.paymentid").alias("paymentid"),
        F.col("pay.paymentmethod").alias("payment_method"),
        F.col("pay.paymentdate").alias("payment_date"),
        F.col("pay.amount").alias("payment_amount")
    )
)

# 6. Orders + Shipments + Shippers
orders_shipments_shippers = (
    orders.alias("o")
    .join(
        shipments.alias("sh"),
        F.col("o.orderid") == F.col("sh.orderid"),
        "left"
    )
    .join(
        shippers.alias("sp"),
        F.col("sh.shipperid") == F.col("sp.shipperid"),
        "left"
    )
    .select(
        F.col("o.orderid").alias("orderid"),
        F.col("o.customerid").alias("customerid"),
        F.col("o.orderdate").alias("orderdate"),
        F.col("o.status").alias("status"),
        F.col("sh.shipmentid").alias("shipmentid"),
        F.col("sh.shipdate").alias("ship_date"),
        F.col("sh.deliverydate").alias("delivery_date"),
        F.col("sp.shipperid").alias("shipperid"),
        F.col("sp.companyname").alias("shipper_name")
    )
)

print("Joins completed successfully.")

print("Orders + Customers:", orders_customers.count())
print("Sales dataset:", sales_dataset.count())
print("Products + Categories:", products_categories.count())
print("Products + Suppliers:", products_suppliers.count())
print("Orders + Payments:", orders_payments.count())
print("Orders + Shipments + Shippers:", orders_shipments_shippers.count())


Joins completed successfully.
Orders + Customers: 50000
Sales dataset: 100000
Products + Categories: 1000
Products + Suppliers: 2027
Orders + Payments: 65333
Orders + Shipments + Shippers: 62541


In [22]:
# Part 7 - Aggregations

from pyspark.sql import functions as F

# Overall metrics

# First calculate one total for each order
order_level_sales = (
    fact_sales
    .groupBy("orderid")
    .agg(
        F.sum("total_amount")
         .cast("decimal(14,2)")
         .alias("order_total"),

        F.sum("quantity")
         .cast("long")
         .alias("order_quantity")
    )
)

# Then calculate overall metrics
overall_metrics = (
    fact_sales
    .agg(
        F.sum("total_amount")
         .cast("decimal(14,2)")
         .alias("total_sales"),

        F.countDistinct("orderid")
         .alias("total_orders"),

        F.sum("quantity")
         .cast("long")
         .alias("total_quantity_sold")
    )
    .crossJoin(
        order_level_sales.agg(
            F.avg("order_total")
             .cast("decimal(14,2)")
             .alias("average_order_value")
        )
    )
)

print("Overall metrics:")
overall_metrics.show(truncate=False)

# Sales by customer
customer_sales = (
    fact_sales
    .groupBy("customerid", "customer_name", "city", "country")
    .agg(
        F.sum("total_amount")
         .cast("decimal(14,2)")
         .alias("total_sales"),

        F.countDistinct("orderid")
         .alias("order_count"),

        F.sum("quantity")
         .cast("long")
         .alias("total_quantity")
    )
    .withColumn(
        "customer_segment",
        F.when(F.col("total_sales") >= 10000, "VIP")
         .when(F.col("total_sales") >= 5000, "PREMIUM")
         .when(F.col("total_sales") >= 1000, "REGULAR")
         .otherwise("LOW_VALUE")
    )
)

# Sales by product
product_sales = (
    fact_sales
    .groupBy("productid", "productname")
    .agg(
        F.sum("total_amount")
         .cast("decimal(14,2)")
         .alias("total_sales"),

        F.sum("quantity")
         .cast("long")
         .alias("total_quantity_sold"),

        F.countDistinct("orderid")
         .alias("order_count")
    )
)

# Sales by category
category_sales = (
    fact_sales
    .groupBy("categoryid", "categoryname")
    .agg(
        F.sum("total_amount")
         .cast("decimal(14,2)")
         .alias("total_sales"),

        F.sum("quantity")
         .cast("long")
         .alias("total_quantity_sold"),

        F.countDistinct("orderid")
         .alias("order_count")
    )
)

# Sales by country
country_sales = (
    fact_sales
    .groupBy("country")
    .agg(
        F.sum("total_amount")
         .cast("decimal(14,2)")
         .alias("total_sales"),

        F.countDistinct("orderid")
         .alias("order_count")
    )
)

# Sales by city
city_sales = (
    fact_sales
    .groupBy("city")
    .agg(
        F.sum("total_amount")
         .cast("decimal(14,2)")
         .alias("total_sales"),

        F.countDistinct("orderid")
         .alias("order_count")
    )
)

# Monthly sales
monthly_sales = (
    fact_sales
    .groupBy("year", "month")
    .agg(
        F.sum("total_amount")
         .cast("decimal(14,2)")
         .alias("total_sales"),

        F.countDistinct("orderid")
         .alias("order_count"),

        F.sum("quantity")
         .cast("long")
         .alias("total_quantity_sold")
    )
    .orderBy("year", "month")
)

# Yearly sales
yearly_sales = (
    fact_sales
    .groupBy("year")
    .agg(
        F.sum("total_amount")
         .cast("decimal(14,2)")
         .alias("total_sales"),

        F.countDistinct("orderid")
         .alias("order_count"),

        F.sum("quantity")
         .cast("long")
         .alias("total_quantity_sold")
    )
    .orderBy("year")
)

# Sales by order status
status_sales = (
    fact_sales
    .groupBy("status")
    .agg(
        F.sum("total_amount")
         .cast("decimal(14,2)")
         .alias("total_sales"),

        F.countDistinct("orderid")
         .alias("order_count"),

        F.sum("quantity")
         .cast("long")
         .alias("total_quantity_sold")
    )
    .orderBy("status")
)

# Number of orders per customer
orders_per_customer = (
    orders
    .groupBy("customerid")
    .agg(
        F.countDistinct("orderid").alias("order_count")
    )
)

print("Aggregation datasets created successfully.")

print("customer_sales rows:", customer_sales.count())
print("product_sales rows:", product_sales.count())
print("category_sales rows:", category_sales.count())
print("monthly_sales rows:", monthly_sales.count())

print("Top 5 categories by sales:")
category_sales.orderBy(F.col("total_sales").desc()).show(5, truncate=False)

print("Monthly sales:")
monthly_sales.show(10, truncate=False)


Overall metrics:
+------------+------------+-------------------+-------------------+
|total_sales |total_orders|total_quantity_sold|average_order_value|
+------------+------------+-------------------+-------------------+
|840507631.75|43136       |549473             |19485.06           |
+------------+------------+-------------------+-------------------+

Aggregation datasets created successfully.
customer_sales rows: 9859
product_sales rows: 1000
category_sales rows: 20
monthly_sales rows: 33
Top 5 categories by sales:
+----------+------------+-----------+-------------------+-----------+
|categoryid|categoryname|total_sales|total_quantity_sold|order_count|
+----------+------------+-----------+-------------------+-----------+
|12        |Gaming      |57306656.35|37546              |6437       |
|1         |Electronics |56420102.49|35271              |6020       |
|6         |Clothing    |52495546.95|28916              |4963       |
|19        |Jewelry     |49331847.38|28486            

In [23]:
# Part 8 - Advanced Analytics

from pyspark.sql import functions as F

# 1. Top 10 customers by sales
top_10_customers = (
    customer_sales
    .orderBy(F.col("total_sales").desc())
    .limit(10)
)

# 2. Top 10 products by sales
top_10_products = (
    product_sales
    .orderBy(F.col("total_sales").desc())
    .limit(10)
)

# 3. Top 5 categories by sales
top_5_categories = (
    category_sales
    .orderBy(F.col("total_sales").desc())
    .limit(5)
)

# 4. Customers who have never placed an order
customers_never_ordered = (
    customers
    .select("customerid", "full_name", "city", "country")
    .join(
        orders.select("customerid").distinct(),
        on="customerid",
        how="left_anti"
    )
)

# 5. Products that have never been ordered
products_never_ordered = (
    products
    .select("productid", "productname", "categoryid", "price")
    .join(
        order_details.select("productid").distinct(),
        on="productid",
        how="left_anti"
    )
)

# 6. Customers with more than 10 orders
customers_more_than_10_orders = (
    orders
    .groupBy("customerid")
    .agg(
        F.countDistinct("orderid").alias("order_count")
    )
    .filter(F.col("order_count") > 10)
    .join(
        customers.select("customerid", "full_name"),
        on="customerid",
        how="left"
    )
    .select("customerid", "full_name", "order_count")
    .orderBy(F.col("order_count").desc())
)

# 7. Calculate order totals at order level
order_totals_for_checks = (
    fact_sales
    .groupBy("orderid")
    .agg(
        F.sum("total_amount")
         .cast("decimal(14,2)")
         .alias("calculated_order_total")
    )
)

# 8. Aggregate payments by order before comparing
payment_totals_for_checks = (
    payments
    .groupBy("orderid")
    .agg(
        F.sum("amount")
         .cast("decimal(14,2)")
         .alias("paid_amount")
    )
)

# 9. Orders where payment amount does not match calculated order total
payment_mismatches = (
    order_totals_for_checks
    .join(
        payment_totals_for_checks,
        on="orderid",
        how="left"
    )
    .withColumn(
        "difference",
        F.round(
            F.col("paid_amount") - F.col("calculated_order_total"),
            2
        )
    )
    .filter(
        F.col("paid_amount").isNull() |
        (F.abs(F.col("difference")) > F.lit(0.01))
    )
)

# 10. Orders that do not have a shipment
orders_without_shipment = (
    orders
    .select("orderid", "customerid", "orderdate", "status")
    .join(
        shipments.select("orderid").distinct(),
        on="orderid",
        how="left_anti"
    )
)

# 11. Shipments without a matching order
shipments_without_order = (
    shipments
    .join(
        orders.select("orderid").distinct(),
        on="orderid",
        how="left_anti"
    )
)

print("Advanced analytics completed successfully.")

print("Top 10 customers:")
top_10_customers.show(10, truncate=False)

print("Top 10 products:")
top_10_products.show(10, truncate=False)

print("Top 5 categories:")
top_5_categories.show(5, truncate=False)

print("Customers never ordered:", customers_never_ordered.count())
print("Products never ordered:", products_never_ordered.count())
print("Customers with more than 10 orders:",
      customers_more_than_10_orders.count())
print("Payment mismatches:", payment_mismatches.count())
print("Orders without shipment:", orders_without_shipment.count())
print("Shipments without matching order:",
      shipments_without_order.count())


Advanced analytics completed successfully.
Top 10 customers:
+----------+----------------+---------+--------------+-----------+-----------+--------------+----------------+
|customerid|customer_name   |city     |country       |total_sales|order_count|total_quantity|customer_segment|
+----------+----------------+---------+--------------+-----------+-----------+--------------+----------------+
|6772      |Angela Stewart  |Abu Dhabi|United Kingdom|368984.90  |12         |211           |VIP             |
|245       |Amy Morrison    |Berlin   |Jordan        |338329.93  |9          |192           |VIP             |
|9895      |Benjamin Holder |Jeddah   |France        |330803.47  |7          |198           |VIP             |
|5446      |Lisa Hogan      |Giza     |Jordan        |321600.74  |9          |184           |VIP             |
|5688      |Jeffrey Ramirez |Riyadh   |Egypt         |319630.49  |11         |169           |VIP             |
|4971      |Michael Russell |London   |Jordan      

In [24]:
# Part 9 - Date Analysis

from pyspark.sql import functions as F

# 1. Daily sales
daily_sales = (
    fact_sales
    .groupBy("orderdate")
    .agg(
        F.sum("total_amount")
         .cast("decimal(14,2)")
         .alias("total_sales"),

        F.countDistinct("orderid")
         .alias("order_count"),

        F.sum("quantity")
         .cast("long")
         .alias("total_quantity_sold")
    )
    .orderBy("orderdate")
)

# 2. Monthly sales
monthly_sales_date = (
    fact_sales
    .groupBy(
        F.year("orderdate").alias("year"),
        F.month("orderdate").alias("month")
    )
    .agg(
        F.sum("total_amount")
         .cast("decimal(14,2)")
         .alias("total_sales"),

        F.countDistinct("orderid")
         .alias("order_count"),

        F.sum("quantity")
         .cast("long")
         .alias("total_quantity_sold")
    )
    .orderBy("year", "month")
)

# 3. Quarterly sales
quarterly_sales = (
    fact_sales
    .groupBy(
        F.year("orderdate").alias("year"),
        F.quarter("orderdate").alias("quarter")
    )
    .agg(
        F.sum("total_amount")
         .cast("decimal(14,2)")
         .alias("total_sales"),

        F.countDistinct("orderid")
         .alias("order_count"),

        F.sum("quantity")
         .cast("long")
         .alias("total_quantity_sold")
    )
    .orderBy("year", "quarter")
)

# 4. Yearly sales
yearly_sales_date = (
    fact_sales
    .groupBy(F.year("orderdate").alias("year"))
    .agg(
        F.sum("total_amount")
         .cast("decimal(14,2)")
         .alias("total_sales"),

        F.countDistinct("orderid")
         .alias("order_count"),

        F.sum("quantity")
         .cast("long")
         .alias("total_quantity_sold")
    )
    .orderBy("year")
)

# 5. Month with the highest sales
highest_sales_month = (
    monthly_sales_date
    .orderBy(F.col("total_sales").desc())
    .limit(1)
)

# 6. Day with the highest number of orders
# Use orders, not fact_sales, so order details do not duplicate the count.
daily_order_counts = (
    orders
    .groupBy("orderdate")
    .agg(
        F.countDistinct("orderid").alias("order_count")
    )
)

highest_order_day = (
    daily_order_counts
    .orderBy(
        F.col("order_count").desc(),
        F.col("orderdate").asc()
    )
    .limit(1)
)

# 7. Average number of orders per month
orders_by_month = (
    orders
    .groupBy(
        F.year("orderdate").alias("year"),
        F.month("orderdate").alias("month")
    )
    .agg(
        F.countDistinct("orderid").alias("order_count")
    )
)

average_orders_per_month = (
    orders_by_month
    .agg(
        F.avg("order_count")
         .cast("decimal(14,2)")
         .alias("average_orders_per_month")
    )
)

print("Date analysis completed successfully.")

print("Daily sales rows:", daily_sales.count())
print("Monthly sales rows:", monthly_sales_date.count())
print("Quarterly sales rows:", quarterly_sales.count())
print("Yearly sales rows:", yearly_sales_date.count())

print("Month with highest sales:")
highest_sales_month.show(truncate=False)

print("Day with highest number of orders:")
highest_order_day.show(truncate=False)

print("Average orders per month:")
average_orders_per_month.show(truncate=False)


Date analysis completed successfully.
Daily sales rows: 989
Monthly sales rows: 33
Quarterly sales rows: 11
Yearly sales rows: 3
Month with highest sales:
+----+-----+-----------+-----------+-------------------+
|year|month|total_sales|order_count|total_quantity_sold|
+----+-----+-----------+-----------+-------------------+
|2025|3    |27889327.64|1389       |18069              |
+----+-----+-----------+-----------+-------------------+

Day with highest number of orders:
+----------+-----------+
|orderdate |order_count|
+----------+-----------+
|2024-09-10|75         |
+----------+-----------+

Average orders per month:
+------------------------+
|average_orders_per_month|
+------------------------+
|1515.15                 |
+------------------------+


In [25]:
# Part 10 - Write final outputs as Parquet with Snappy

OUTPUT_PATH = f"s3://{BUCKET}/processed/ecommerce"

def write_parquet(df, path, partition_cols=None):
    writer = (
        df.write
        .mode("overwrite")
        .format("parquet")
        .option("compression", "snappy")
    )

    if partition_cols:
        writer = writer.partitionBy(*partition_cols)

    writer.save(path)

# 1. Cleaned base datasets
write_parquet(customers, f"{OUTPUT_PATH}/customers")
write_parquet(products, f"{OUTPUT_PATH}/products")
write_parquet(categories, f"{OUTPUT_PATH}/categories")

# Orders must be partitioned by year and month
write_parquet(
    orders,
    f"{OUTPUT_PATH}/orders",
    partition_cols=["year", "month"]
)

write_parquet(order_details, f"{OUTPUT_PATH}/order_details")
write_parquet(payments, f"{OUTPUT_PATH}/payments")
write_parquet(shipments, f"{OUTPUT_PATH}/shipments")

# 2. Analytical datasets
write_parquet(fact_sales, f"{OUTPUT_PATH}/fact_sales")
write_parquet(customer_sales, f"{OUTPUT_PATH}/customer_sales")
write_parquet(product_sales, f"{OUTPUT_PATH}/product_sales")
write_parquet(category_sales, f"{OUTPUT_PATH}/category_sales")
write_parquet(monthly_sales, f"{OUTPUT_PATH}/monthly_sales")

print("All required datasets were written successfully.")
print(f"Output path: {OUTPUT_PATH}")


All required datasets were written successfully.
Output path: s3://mahmoud-sic-ecommerce-2026/processed/ecommerce


In [36]:
customers.show(10, truncate=False)
orders.show(10, truncate=False)

+----------+---------+---------+-----------------------------+----------+-------------+-------------------+
|CustomerID|FirstName|LastName |Email                        |City      |Country      |RegistrationDate   |
+----------+---------+---------+-----------------------------+----------+-------------+-------------------+
|1         |Kristen  |Roberts  |kristen.roberts1@example.com |Abu Dhabi |United States|2026-09-13 00:00:00|
|2         |Sara     |Gill     |sara.gill2@example.com       |Berlin    |United States|2026-01-10 00:00:00|
|3         |Cody     |Burns    |cody.burns3@example.com      |Amman     |France       |2026-02-09 00:00:00|
|4         |John     |Anderson |john.anderson4@example.com   |Abu Dhabi |Jordan       |2021-11-14 00:00:00|
|5         |Jeanette |Fuller   |jeanette.fuller5@example.com |Giza      |United States|2026-09-07 00:00:00|
|6         |Dylan    |Fernandez|dylan.fernandez6@example.com |Berlin    |Saudi Arabia |2020-11-02 00:00:00|
|7         |Lori     |Griffi

In [37]:
def clean_column_names(df):
    for column in df.columns:
        new_name = (
            column.strip()
            .lower()
            .replace(" ", "_")
            .replace("-", "_")
        )
        
        df = df.withColumnRenamed(column, new_name)
    
    return df

In [38]:
customers = clean_column_names(customers)
categories = clean_column_names(categories)
products = clean_column_names(products)
departments = clean_column_names(departments)
employees = clean_column_names(employees)
suppliers = clean_column_names(suppliers)
orders = clean_column_names(orders)
order_details = clean_column_names(order_details)
payments = clean_column_names(payments)
product_suppliers = clean_column_names(product_suppliers)
shippers = clean_column_names(shippers)
shipments = clean_column_names(shipments)

In [39]:
customers.printSchema()

root
 |-- customerid: integer (nullable = true)
 |-- firstname: string (nullable = true)
 |-- lastname: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- registrationdate: timestamp (nullable = true)


In [40]:
departments.printSchema()

root
 |-- departmentid: integer (nullable = true)
 |-- departmentname: string (nullable = true)


In [41]:
customers = customers.dropDuplicates(["customerid"])

orders = orders.dropDuplicates(["orderid"])

products = products.dropDuplicates(["productid"])

order_details = order_details.dropDuplicates(["orderdetailid"])

In [14]:
customers.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in customers.columns
]).show()

+----------+---------+--------+-----+----+-------+----------------+
|customerid|firstname|lastname|email|city|country|registrationdate|
+----------+---------+--------+-----+----+-------+----------------+
|         0|        0|       0|    0|   0|      0|               0|
+----------+---------+--------+-----+----+-------+----------------+


In [42]:
customers = (
    customers
    .withColumn("firstname", F.trim("firstname"))
    .withColumn("lastname", F.trim("lastname"))
    .withColumn("email", F.lower(F.trim("email")))
    .withColumn("city", F.trim("city"))
    .withColumn("country", F.trim("country"))
)

In [43]:
orders = (
    orders
    .withColumn("status", F.upper(F.trim("status")))
)

In [45]:
orders = (
    orders
    .withColumn(
        "orderid",
        F.col("orderid").cast("long")
    )
    .withColumn(
        "customerid",
        F.col("customerid").cast("long")
    )
    .withColumn(
        "orderdate",
        F.to_date("orderdate")
    )
)

In [46]:
customers = customers.withColumn(
    "valid_email",
    F.when(
        F.col("email").rlike(
            r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
        ),
        True
    ).otherwise(False)
)

In [47]:
orders = orders.withColumn(
    "status",
    F.when(
        F.col("status").isin(
            "PENDING",
            "SHIPPED",
            "DELIVERED",
            "CANCELLED"
        ),
        F.col("status")
    ).otherwise("UNKNOWN")
)

In [48]:
order_details = order_details.withColumn(
    "total_amount",
    F.col("quantity") * F.col("unitprice")
)

In [49]:
order_details = order_details.withColumn(
    "quantity",
    F.when(
        F.col("quantity") < 0,
        0
    ).otherwise(F.col("quantity"))
)

In [50]:
order_details = order_details.withColumn(
    "unitprice",
    F.when(
        F.col("unitprice") < 0,
        0
    ).otherwise(F.col("unitprice"))
)

In [51]:
orders = (
    orders
    .withColumn("year", F.year("orderdate"))
    .withColumn("month", F.month("orderdate"))
    .withColumn("day", F.dayofmonth("orderdate"))
    .withColumn("quarter", F.quarter("orderdate"))
    .withColumn("day_of_week", F.dayofweek("orderdate"))
)

In [52]:
order_sales = (
    orders.alias("o")
    .join(
        order_details.alias("od"),
        F.col("o.orderid") == F.col("od.orderid"),
        "inner"
    )
)

In [53]:
order_sales = order_sales.select(
    F.col("o.orderid"),
    F.col("o.customerid"),
    F.col("o.orderdate"),
    F.col("o.status"),
    F.col("od.productid"),
    F.col("od.quantity"),
    F.col("od.unitprice"),
    F.col("od.total_amount")
)

In [27]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Define the window
customer_window = (
    Window
    .partitionBy("customerid")
    .orderBy(F.col("orderdate").desc())
)

# Add row number
last_order_customer = (
    orders
    .withColumn(
        "last_order_rank",
        F.row_number().over(customer_window)
    )
)

# Keep only the latest order for each customer
last_order_customer = (
    last_order_customer
    .filter(F.col("last_order_rank") == 1)
)

last_order_customer.show()

+-------+----------+----------+---------+----+-----+---+-------+-----------+---------------+
|orderid|customerid| orderdate|   status|year|month|day|quarter|day_of_week|last_order_rank|
+-------+----------+----------+---------+----+-----+---+-------+-----------+---------------+
| 960221|        28|2024-11-06|  PENDING|2024|   11|  6|      4|          4|              1|
| 686734|        29|2026-06-29|  SHIPPED|2026|    6| 29|      2|          2|              1|
| 153064|        30|2026-07-11|DELIVERED|2026|    7| 11|      3|          7|              1|
|1984435|        33|2025-11-26|DELIVERED|2025|   11| 26|      4|          4|              1|
|1324387|        93|2026-06-15|  SHIPPED|2026|    6| 15|      2|          2|              1|
| 822306|       151|2026-04-23|  SHIPPED|2026|    4| 23|      2|          5|              1|
|1301514|       171|2025-09-21|  UNKNOWN|2025|    9| 21|      3|          1|              1|
| 903362|       200|2026-08-09|  UNKNOWN|2026|    8|  9|      3|      

In [54]:
spark.stop()